![Roboflow Notebooks banner](https://camo.githubusercontent.com/aec53c2b5fb6ed43d202a0ab622b58ba68a89d654fbe3abab0c0cc8bd1ff424e/68747470733a2f2f696b2e696d6167656b69742e696f2f726f626f666c6f772f6e6f7465626f6f6b732f74656d706c6174652f62616e6e657274657374322d322e706e673f696b2d73646b2d76657273696f6e3d6a6176617363726970742d312e342e33267570646174656441743d31363732393332373130313934)

# Image Classification with DINOv2

DINOv2, released by Meta Research in April 2023, implements a self-supervised method of training computer vision models.

DINOv2 was trained using 140 million images without labels. The embeddings generated by DINOv2 can be used for classification, image retrieval, segmentation, and depth estimation. With that said, Meta Research did not release heads for segmentation and depth estimation.

In this guide, we are going to build an image classifier using embeddings from DINOv2. To do so, we will:

1. Load a folder of images
2. Compute embeddings for each image
3. Save all the embeddings in a file and vector store
4. Train an SVM classifier to classify images

We'll be using the [MIT Indoor Scene Recognition dataset](https://universe.roboflow.com/popular-benchmarks/mit-indoor-scene-recognition/) in this project, but you can use any labelled classification dataset you have.

By the end of this notebook, we'll have a classifier trained on our dataset.

Without further ado, let's begin!

## Import Packages

First, let's import the packages we will need for this project.

In [1]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import os
import cv2
import json
import glob
from tqdm.notebook import tqdm

## Load Data

In this guide, we're going to work with the [MIT Indoor Scene Recognition dataset](https://universe.roboflow.com/popular-benchmarks/mit-indoor-scene-recognition), hosted on Roboflow Universe. To download this dataset, you will need a [free Roboflow account](https://app.roboflow.com).

Let's download the dataset and create a dictionary that maps each image in our training dataset to its associated label.

In [2]:
!pip install roboflow supervision -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 125.6 MB/s eta 0:00:00


In [6]:
import roboflow
import supervision as sv

roboflow.login()

rf = roboflow.Roboflow()

project = rf.workspace("qsa-cycling-recon").project("helico_view")
dataset = project.version(2).download("folder")

visit https://app.roboflow.com/auth-cli to get your authentication token.
Paste the authentication token here: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Helico_View-2 in folder:: 100%|██████████| 1250/1250 [00:00<00:00, 3534.56it/s]


In [7]:
cwd = os.getcwd()

ROOT_DIR = os.path.join(cwd, "Helico_View-2/train")

labels = {}

for folder in os.listdir(ROOT_DIR):
    for file in os.listdir(os.path.join(ROOT_DIR, folder)):
        if file.endswith(".jpg"):
            full_name = os.path.join(ROOT_DIR, folder, file)
            labels[full_name] = folder

files = labels.keys()

## Load the Model and Compute Embeddings

To train our classifier, we need:

1. The embeddings associated with each image in our dataset, and;
2. The labels associated with each image.

To calculate embeddings, we'll use DINOv2. Below, we load the smallest DINOv2 weights and define functions that will load and compute embeddings for every image in a specified list.

We store all of our vectors in a dictionary that is saved to disk so we can reference them again if needed. Note that in production environments one may opt for using another data structure such as a vector embedding database (i.e. faiss) for storing embeddings.

In [8]:
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

dinov2_vits14.to(device)

transform_image = T.Compose([T.ToTensor(), T.Resize(244), T.CenterCrop(224), T.Normalize([0.5], [0.5])])

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 337MB/s]


In [9]:
def load_image(img: str) -> torch.Tensor:
    """
    Load an image and return a tensor that can be used as an input to DINOv2.
    """
    img = Image.open(img)

    transformed_img = transform_image(img)[:3].unsqueeze(0)

    return transformed_img

def compute_embeddings(files: list) -> dict:
    """
    Create an index that contains all of the images in the specified list of files.
    """
    all_embeddings = {}

    with torch.no_grad():
      for i, file in enumerate(tqdm(files)):
        embeddings = dinov2_vits14(load_image(file).to(device))

        all_embeddings[file] = np.array(embeddings[0].cpu().numpy()).reshape(1, -1).tolist()

    with open("all_embeddings.json", "w") as f:
        f.write(json.dumps(all_embeddings))

    return all_embeddings

## Compute Embeddings

The code below computes the embeddings for all the images in our dataset. This step will take a few minutes for the MIT Indoor Scene Recognition dataset. There are over 10,000 images in the training set that we need to pass through DINOv2.

In [10]:
embeddings = compute_embeddings(files)

  0%|          | 0/1080 [00:00<?, ?it/s]

## Train a Classification Model

The embeddings we have computed can be used as an input in a classification model. For this guide, we will be using SVM, a linear classification model.

Below, we make lists of both all of the embeddings we have computed and their associated labels. We then fit our model using those lists.

In [11]:
from sklearn import svm

clf = svm.SVC(gamma='scale')

y = [labels[file] for file in files]

embedding_list = list(embeddings.values())

clf.fit(np.array(embedding_list).reshape(-1, 384), y)

SVC()

## Classify an Image

We now have a classifier we can use to classify images!

Change the `input_file` value below to the path of a file in the `valid` or `test` directories in the image dataset with which we have been working.

Then, run the cell to classify the image.

In [14]:
ls Helico_View-2/test/Helico

frame_0058_png.rf.8812dfbdf813d293ac81ec23e126a0fd.jpg
frame_0064_png.rf.c055e0b26e002ee91bced5fa67cde56c.jpg
frame_0144_png.rf.cfa88d1bb9a63c11e274a50c5ae0992c.jpg
frame_0154_png.rf.ea68dc89d6cf2e1abda7021876e6ffd0.jpg
frame_0156_png.rf.5446fc096550caa6de1168922f37f5d2.jpg
frame_0203_png.rf.0942a96d676038cc7a4b7ffb884817e0.jpg
frame_0205_png.rf.a5537486568596b25e2a8b8f849d7cc5.jpg
frame_0207_png.rf.ccb92e79dafe268cc23562b22c04dded.jpg
frame_0305_png.rf.c5162ab9c85acad9e1a7bf9b930d401a.jpg
frame_0309_png.rf.20653bca5feb5196af9238363ce3da22.jpg
frame_0310_png.rf.c5bc2773e1c4e9ecd053fe67a819ec87.jpg
frame_0350_png.rf.4abd2cfeef69d0cfca7e8c21b09d19b9.jpg
frame_0351_png.rf.c794bd8bfb95d8d940d709bd5ff5f857.jpg
frame_0356_png.rf.23bb765e6ea936d88f11f128a3f6f252.jpg
frame_0364_png.rf.b3c9b65befda6d8e0717f9f9eadf9dc1.jpg
frame_0375_png.rf.6a1500bb290cbc7039a4325f2e8fb732.jpg
frame_0378_png.rf.47cdc381a2560737fd8df6dc2ff038f5.jpg
frame_0416_png.rf.9bd2417c7934a91ca5bc20d4476f562b.jpg
PN_Sprint_

In [15]:
folder_path_helico = 'Helico_View-2/test/Helico'
test_helico_files = [os.path.join(folder_path_helico, file) for file in os.listdir(folder_path_helico)]

folder_path_moto = 'Helico_View-2/test/Moto'
test_moto_files = [os.path.join(folder_path_moto, file) for file in os.listdir(folder_path_moto)]

In [17]:
for input_file in test_helico_files:
  new_image = load_image(input_file)
  %matplotlib inline
  #sv.plot_image(image=cv2.imread(input_file), size=(8, 8))

  with torch.no_grad():
      embedding = dinov2_vits14(new_image.to(device))

      prediction = clf.predict(np.array(embedding[0].cpu()).reshape(1, -1))

      #print()
      print("Predicted class: " + prediction[0])

Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico
Predicted class: Helico


## 🏆 Congratulations

### Learning Resources

Roboflow has produced many resources that you may find interesting as you advance your knowledge of computer vision:

- [Roboflow Notebooks](https://github.com/roboflow/notebooks): A repository of over 20 notebooks that walk through how to train custom models with a range of model types, from YOLOv7 to SegFormer.
- [Roboflow YouTube](https://www.youtube.com/c/Roboflow): Our library of videos featuring deep dives into the latest in computer vision, detailed tutorials that accompany our notebooks, and more.
- [Roboflow Discuss](https://discuss.roboflow.com/): Have a question about how to do something on Roboflow? Ask your question on our discussion forum.
- [Roboflow Models](https://roboflow.com): Learn about state-of-the-art models and their performance. Find links and tutorials to guide your learning.

### Convert data formats

Roboflow provides free utilities to convert data between dozens of popular computer vision formats. Check out [Roboflow Formats](https://roboflow.com/formats) to find tutorials on how to convert data between formats in a few clicks.

### Connect computer vision to your project logic

[Roboflow Templates](https://roboflow.com/templates) is a public gallery of code snippets that you can use to connect computer vision to your project logic. Code snippets range from sending emails after inference to measuring object distance between detections.

In [72]:
import joblib
joblib.dump(clf, 'helico_view.pkl')

['helico_view.pkl']

In [75]:
from google.colab import files
files.download('helico_view.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
clf

SVC()